# STIR-Net V1 — 21 spatial-fix micro experiments

Notebook 20 localized the current easy-merge failure to the **spatial instance representation**:

- the failure is already present at the end of the 30-step spatial-only stage;
- source-9 internal cell-cell boundaries are poorly represented by the trained dense boundary output;
- internal cell-cell boundaries are a very small fraction of the current mixed boundary target;
- E2 contains more separability than D1/D0 preserve;
- oracle centers and much larger query-token budgets do not rescue masks;
- the coarse spatial feature space is not sufficiently instance-separable.

This notebook does **not** implement a permanent source patch yet.

Instead, it runs several short, independent mechanism experiments from the exact same step-30 checkpoint to determine **which fix produces the fastest causal movement in the failed metrics**.

## Runtime strategy

The expensive encoder and `stage_e2` are executed **once** and detached.

Every experiment trains only:

```text
cached E2
  ↓
stage_e1
  ↓
D1
  ↓
stage_e0
  ↓
D0
  ↓
existing dense heads
```

This keeps the experiment fast and asks a focused question:

> Given the useful signal already present at E2, what supervision makes D1/D0 preserve and expose individual touching-cell identities?

There is:

- no temporal branch;
- no CR;
- no query decoder;
- no Hungarian matching;
- no native mask rendering;
- no full overfit.

## Independent experiment arms

All arms start from the **same step-30 weights**.

1. **baseline**  
   Continue the current dense objective unchanged.

2. **internal_boundary**  
   Current objective + independently normalized cell-cell boundary supervision on the existing D0 boundary head.

3. **deep_internal_boundary**  
   Arm 2 + a D1 auxiliary internal-boundary head. This tests whether deep supervision specifically prevents the E2 → D1 information loss.

4. **instance_prototype**  
   Current objective + instance-prototype contrastive losses on D1 and D0 for every merged current component. This directly asks the feature space to distinguish neighboring GT cells.

5. **max_center_target**  
   Replace the current summed/smoothed center target with a max-composed per-cell Gaussian target.

6. **combined**  
   Internal boundary + deep D1 boundary + D1/D0 instance prototypes + max-composed center target.

## Funnel

```text
target sanity
   ↓
zero-step common baseline
   ↓
3-step independent screen for all 6 arms
   ↓
rank by failed spatial metrics
   ↓
extend only the best 2 non-baseline arms to step 5
   ↓
GREEN / YELLOW / RED mechanism decision
```

The notebook intentionally stops at five tail-training steps.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import resize_label_map_nearest
from learned.stirnet.training.checkpoint import load_checkpoint

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

# Fast mechanism screen.
SCREEN_STEPS = 3
EXTEND_TO_STEP = 5
EVAL_STEPS = {0, 1, 3}

# Same LR for every arm. This is a mechanism test, not the final recipe.
MICRO_LR = 5e-4

# New-objective weights. Keep them moderate; every arm uses the same base loss.
LAMBDA_INTERNAL_D0 = 1.0
LAMBDA_INTERNAL_D1 = 0.5
LAMBDA_PROTO_D1 = 0.5
LAMBDA_PROTO_D0 = 0.5

# Fixed sampled supervision sizes.
MAX_INTERNAL_POS = 8192
INTERNAL_NEG_RATIO = 1.0
PROTO_HALF_PER_CLASS = 128
PROTO_TEMPERATURE = 0.15

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

STEP30_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "21_spatial_fix_micro_experiments"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 21 requires CUDA.")

if not STEP30_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Required exact step-30 spatial checkpoint not found:\n"
        f"{STEP30_CHECKPOINT}"
    )

device = torch.device("cuda")
cfg = _reduced_config()

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", STEP30_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Reduced spatial channels:", cfg.spatial.channels)
print("Micro LR   :", MICRO_LR)

# 1. Load the exact full scene

The whole biological scene remains the base training sample. No isolated-cell patch dataset is constructed.

Only the new instance-specific auxiliary losses use bounded samples from merged components.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
target = batch["targets"][0]
targets_original = batch["targets"]

spatial_inputs = batch["spatial_inputs"].to(
    device=device,
    dtype=AMP_DTYPE,
    non_blocking=True,
)
spacing_um = batch["spacing_um"].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)
dref_tensor = batch["dref_um"].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)

spatial_padding_mask = batch.get("spatial_padding_mask")
if spatial_padding_mask is not None:
    spatial_padding_mask = spatial_padding_mask.to(
        device=device,
        non_blocking=True,
    )

current_labels_cpu = (
    batch["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

gt_labels_cpu = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

spacing_native = (
    batch["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

dref_um = float(
    batch["dref_um"][0]
)

target_ids = (
    torch.as_tensor(target["ids"])
    .detach()
    .cpu()
    .long()
)

target_centers_um = (
    torch.as_tensor(
        target.get(
            "centers_um",
            target["centers_cellscale"],
        )
    )
    .detach()
    .cpu()
    .float()
)

if "centers_um" not in target:
    target_centers_um = (
        target_centers_um
        * dref_um
    )

id_to_target_row = {
    int(gt_id): row
    for row, gt_id
    in enumerate(target_ids.tolist())
}

source9_mask_native = (
    current_labels_cpu
    == SOURCE_ID
)

source9_gt_ids = np.unique(
    gt_labels_cpu[
        source9_mask_native
    ]
)
source9_gt_ids = (
    source9_gt_ids[
        source9_gt_ids > 0
    ]
    .astype(int)
)

source9_gt_centers_um = torch.stack([
    target_centers_um[
        id_to_target_row[
            int(gt_id)
        ]
    ]
    for gt_id in source9_gt_ids
])

print(json.dumps(sample, indent=2, default=float))
print("Source-9 GT IDs:", source9_gt_ids.tolist())
print("Source-9 GT count:", len(source9_gt_ids))
print("dref_um:", dref_um)

if len(source9_gt_ids) != 9:
    print(
        "WARNING: expected the canonical 9-cell source-9 merge, "
        f"found {len(source9_gt_ids)} GT cells."
    )

# 2. Build reusable spatial targets

We explicitly distinguish:

```text
all boundary:
    cell↔background + cell↔cell

internal boundary:
    cell A ↔ cell B only
```

The internal target is physically dilated by 1 µm, matching the current dense boundary width.

In [ ]:
def internal_instance_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros_like(
        labels,
        dtype=bool,
    )

    for axis in range(3):
        left = [slice(None)] * 3
        right = [slice(None)] * 3

        left[axis] = slice(0, -1)
        right[axis] = slice(1, None)

        a = labels[tuple(left)]
        b = labels[tuple(right)]

        diff = (
            (a > 0)
            & (b > 0)
            & (a != b)
        )

        boundary[tuple(left)] |= diff
        boundary[tuple(right)] |= diff

    return boundary


def physical_dilate(mask, spacing, width_um=1.0):
    spacing = np.asarray(
        spacing,
        dtype=np.float64,
    )

    rv = np.ceil(
        float(width_um)
        / spacing
    ).astype(int)

    zz, yy, xx = np.ogrid[
        -rv[0]:rv[0] + 1,
        -rv[1]:rv[1] + 1,
        -rv[2]:rv[2] + 1,
    ]

    structure = (
        (zz * spacing[0]) ** 2
        + (yy * spacing[1]) ** 2
        + (xx * spacing[2]) ** 2
        <= float(width_um) ** 2
    )

    return ndi.binary_dilation(
        mask,
        structure=structure,
    )


internal_raw_native = (
    internal_instance_boundary(
        gt_labels_cpu
    )
)

internal_target_native = (
    physical_dilate(
        internal_raw_native,
        spacing_native,
        width_um=1.0,
    )
)

foreground_native = (
    gt_labels_cpu > 0
)

training_boundary_native = (
    torch.as_tensor(
        target["boundary"]
    )
    .detach()
    .cpu()
    .numpy()
    > 0.5
)

print(
    "All training-boundary positives:",
    int(training_boundary_native.sum()),
)
print(
    "Internal-boundary positives:",
    int(
        (
            internal_target_native
            & training_boundary_native
        ).sum()
    ),
)
print(
    "Internal fraction:",
    float(
        (
            internal_target_native
            & training_boundary_native
        ).sum()
        / max(
            training_boundary_native.sum(),
            1,
        )
    ),
)

# 3. Build the max-composed center target

The current target is produced by filtering all center impulses together.

This alternative constructs one Gaussian per cell locally and combines cells by **maximum**, not summation.

No source file is changed.

In [ ]:
def max_composed_center_heatmap(
    shape,
    centers_um_relative,
    spacing_um,
    sigma_um=2.0,
    truncate_sigma=3.0,
):
    shape = np.asarray(
        shape,
        dtype=np.int64,
    )
    spacing = np.asarray(
        spacing_um,
        dtype=np.float64,
    )

    center_abs_um = (
        0.5
        * (
            shape.astype(
                np.float64
            )
            - 1.0
        )
        * spacing
    )

    result = np.zeros(
        tuple(shape.tolist()),
        dtype=np.float32,
    )

    radius_vox = np.ceil(
        float(truncate_sigma)
        * float(sigma_um)
        / spacing
    ).astype(int)

    for center_rel in np.asarray(
        centers_um_relative,
        dtype=np.float64,
    ):
        center_vox = (
            center_rel
            + center_abs_um
        ) / spacing

        lo = np.maximum(
            np.floor(
                center_vox
            ).astype(int)
            - radius_vox,
            0,
        )
        hi = np.minimum(
            np.floor(
                center_vox
            ).astype(int)
            + radius_vox
            + 2,
            shape,
        )

        z = np.arange(
            lo[0],
            hi[0],
            dtype=np.float64,
        )
        y = np.arange(
            lo[1],
            hi[1],
            dtype=np.float64,
        )
        x = np.arange(
            lo[2],
            hi[2],
            dtype=np.float64,
        )

        dz = (
            (
                z
                - center_vox[0]
            )
            * spacing[0]
        )[:, None, None]

        dy = (
            (
                y
                - center_vox[1]
            )
            * spacing[1]
        )[None, :, None]

        dx = (
            (
                x
                - center_vox[2]
            )
            * spacing[2]
        )[None, None, :]

        gaussian = np.exp(
            -0.5
            * (
                dz * dz
                + dy * dy
                + dx * dx
            )
            / (
                float(sigma_um)
                ** 2
            )
        ).astype(
            np.float32
        )

        # Ensure each cell contributes a unit peak despite subvoxel center position.
        peak = float(
            gaussian.max()
        )
        if peak > 0:
            gaussian /= peak

        view = result[
            lo[0]:hi[0],
            lo[1]:hi[1],
            lo[2]:hi[2],
        ]

        np.maximum(
            view,
            gaussian,
            out=view,
        )

    return result


max_center_target_np = (
    max_composed_center_heatmap(
        gt_labels_cpu.shape,
        target_centers_um.numpy(),
        spacing_native,
        sigma_um=2.0,
    )
)

target_max_center = dict(target)
target_max_center[
    "center_heatmap"
] = torch.from_numpy(
    max_center_target_np
)

targets_max_center = [
    target_max_center
]

print(
    "Original center target max:",
    float(
        torch.as_tensor(
            target["center_heatmap"]
        ).max()
    ),
)
print(
    "Max-composed center target max:",
    float(
        max_center_target_np.max()
    ),
)

# 4. Cache E2 and the high-resolution encoder skips once

This is the key runtime reduction.

`SpatialEncoder + stage_e2` remain fixed at the exact step-30 representation. The experiment only asks whether alternative supervision can teach the tail to preserve/use the E2 instance signal.

In [ ]:
step30_model = StirNet(
    cfg
).to(
    device
)

load_info = load_checkpoint(
    STEP30_CHECKPOINT,
    step30_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

if int(
    load_info.get(
        "step",
        -1,
    )
) != 30:
    raise RuntimeError(
        "Expected the spatial checkpoint payload to be step 30."
    )

step30_model.eval()

torch.cuda.reset_peak_memory_stats()
cache_start = time.perf_counter()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    ACQ_CACHE = (
        step30_model.acquisition(
            spacing_um,
            dref_tensor,
        )
        .detach()
    )

    pyramid = (
        step30_model.encoder(
            spatial_inputs,
            spacing_um,
            ACQ_CACHE,
            spatial_padding_mask,
        )
    )

    E3_CACHE = (
        pyramid.features[3]
        .detach()
    )

    E2_CACHE = (
        step30_model.decoder
        .decode_to_e2(
            E3_CACHE,
            pyramid,
            ACQ_CACHE,
        )
        .detach()
    )

    SKIP1_CACHE = (
        pyramid.features[1]
        .detach()
    )

    SKIP0_CACHE = (
        pyramid.features[0]
        .detach()
    )

    SPACING_E2 = (
        pyramid.spacings_um[2]
        .detach()
        .float()
    )

    SPACING_D1 = (
        pyramid.spacings_um[1]
        .detach()
        .float()
    )

    SPACING_D0 = (
        pyramid.spacings_um[0]
        .detach()
        .float()
    )

cache_seconds = (
    time.perf_counter()
    - cache_start
)

print(
    f"Spatial cache: {cache_seconds:.2f}s"
)
print(
    "E2 / skip1 / skip0:",
    tuple(E2_CACHE.shape),
    tuple(SKIP1_CACHE.shape),
    tuple(SKIP0_CACHE.shape),
)
print(
    "Peak CUDA:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

# Preserve an exact CPU template for independent experiment arms.
step30_model.cpu()

TAIL_TEMPLATE = {
    "stage_e1": copy.deepcopy(
        step30_model.decoder.stage_e1
    ),
    "stage_e0": copy.deepcopy(
        step30_model.decoder.stage_e0
    ),
    "dense_heads": copy.deepcopy(
        step30_model.dense_heads
    ),
}

del step30_model
del pyramid
del spatial_inputs
gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA after dropping encoder model:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        3,
    ),
    "GiB",
)

# 5. Construct D1/D0 targets and fixed bounded samples

In [ ]:
def labels_at_shape(
    labels_native,
    shape,
):
    return (
        resize_label_map_nearest(
            torch.from_numpy(
                labels_native.astype(
                    np.int32,
                    copy=False,
                )
            ),
            tuple(
                int(v)
                for v in shape
            ),
        )
        .cpu()
        .numpy()
        .astype(
            np.int32,
            copy=False,
        )
    )


D1_SHAPE = tuple(
    int(v)
    for v
    in SKIP1_CACHE.shape[-3:]
)

D0_SHAPE = tuple(
    int(v)
    for v
    in SKIP0_CACHE.shape[-3:]
)

GT_D1 = labels_at_shape(
    gt_labels_cpu,
    D1_SHAPE,
)
CURRENT_D1 = labels_at_shape(
    current_labels_cpu,
    D1_SHAPE,
)

GT_D0 = labels_at_shape(
    gt_labels_cpu,
    D0_SHAPE,
)
CURRENT_D0 = labels_at_shape(
    current_labels_cpu,
    D0_SHAPE,
)

INTERNAL_RAW_D1 = (
    internal_instance_boundary(
        GT_D1
    )
)
INTERNAL_RAW_D0 = (
    internal_instance_boundary(
        GT_D0
    )
)

INTERNAL_D1 = physical_dilate(
    INTERNAL_RAW_D1,
    SPACING_D1[
        0
    ].cpu().numpy(),
    width_um=1.0,
)

INTERNAL_D0 = physical_dilate(
    INTERNAL_RAW_D0,
    SPACING_D0[
        0
    ].cpu().numpy(),
    width_um=1.0,
)

FOREGROUND_D1 = (
    GT_D1 > 0
)
FOREGROUND_D0 = (
    GT_D0 > 0
)


def deterministic_choice(
    indices,
    count,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(indices) <= int(count):
        return indices

    rng = np.random.default_rng(
        int(seed)
    )

    selected = rng.choice(
        indices,
        size=int(count),
        replace=False,
    )

    return np.sort(
        selected
    )


@dataclass
class BinarySample:
    indices: torch.Tensor
    targets: torch.Tensor
    positive_count: int
    negative_count: int


def build_internal_sample(
    positive_mask,
    support_mask,
    *,
    max_positive,
    negative_ratio,
    seed,
):
    positive_flat = np.flatnonzero(
        positive_mask.reshape(-1)
    )

    negative_flat = np.flatnonzero(
        (
            support_mask
            & ~positive_mask
        ).reshape(-1)
    )

    positive_flat = deterministic_choice(
        positive_flat,
        min(
            len(positive_flat),
            int(max_positive),
        ),
        seed,
    )

    n_negative = min(
        len(negative_flat),
        int(
            max(
                1,
                round(
                    len(positive_flat)
                    * float(
                        negative_ratio
                    )
                ),
            )
        ),
    )

    negative_flat = deterministic_choice(
        negative_flat,
        n_negative,
        seed + 1,
    )

    indices = np.concatenate(
        [
            positive_flat,
            negative_flat,
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(
                    positive_flat
                ),
                dtype=np.float32,
            ),
            np.zeros(
                len(
                    negative_flat
                ),
                dtype=np.float32,
            ),
        ]
    )

    rng = np.random.default_rng(
        seed + 2
    )
    order = rng.permutation(
        len(indices)
    )

    return BinarySample(
        indices=torch.from_numpy(
            indices[
                order
            ]
        ).long(),
        targets=torch.from_numpy(
            labels[
                order
            ]
        ).float(),
        positive_count=int(
            len(
                positive_flat
            )
        ),
        negative_count=int(
            len(
                negative_flat
            )
        ),
    )


INTERNAL_SAMPLE_D1 = (
    build_internal_sample(
        INTERNAL_D1,
        FOREGROUND_D1,
        max_positive=MAX_INTERNAL_POS,
        negative_ratio=INTERNAL_NEG_RATIO,
        seed=SEED + 101,
    )
)

INTERNAL_SAMPLE_D0 = (
    build_internal_sample(
        INTERNAL_D0,
        FOREGROUND_D0,
        max_positive=MAX_INTERNAL_POS,
        negative_ratio=INTERNAL_NEG_RATIO,
        seed=SEED + 201,
    )
)

print(
    "Internal D1 sample:",
    INTERNAL_SAMPLE_D1.positive_count,
    "+",
    INTERNAL_SAMPLE_D1.negative_count,
)
print(
    "Internal D0 sample:",
    INTERNAL_SAMPLE_D0.positive_count,
    "+",
    INTERNAL_SAMPLE_D0.negative_count,
)

## 5.1 Fixed merged-component instance samples

Prototype supervision is not restricted to source 9.

Every current connected component that overlaps at least two GT cells contributes.

In [ ]:
@dataclass
class PrototypeGroup:
    source_id: int
    gt_ids: tuple
    proto_indices: torch.Tensor
    proto_targets: torch.Tensor
    query_indices: torch.Tensor
    query_targets: torch.Tensor


def build_prototype_groups(
    current_labels,
    gt_labels,
    *,
    half_per_class,
    seed,
):
    current_flat = (
        current_labels.reshape(-1)
    )
    gt_flat = (
        gt_labels.reshape(-1)
    )

    groups = []

    for source_id in np.unique(
        current_flat
    ):
        if source_id <= 0:
            continue

        source_positions = np.flatnonzero(
            current_flat
            == int(
                source_id
            )
        )

        overlap_ids = np.unique(
            gt_flat[
                source_positions
            ]
        )

        overlap_ids = overlap_ids[
            overlap_ids > 0
        ]

        if len(
            overlap_ids
        ) < 2:
            continue

        proto_indices = []
        proto_targets = []
        query_indices = []
        query_targets = []

        valid_ids = []

        for gt_id in overlap_ids:
            positions = np.flatnonzero(
                (
                    current_flat
                    == int(
                        source_id
                    )
                )
                & (
                    gt_flat
                    == int(
                        gt_id
                    )
                )
            )

            if len(
                positions
            ) < 4:
                continue

            rng = np.random.default_rng(
                int(
                    seed
                    + 1009
                    * int(
                        source_id
                    )
                    + 37
                    * int(
                        gt_id
                    )
                )
            )

            positions = positions[
                rng.permutation(
                    len(
                        positions
                    )
                )
            ]

            count_each = min(
                int(
                    half_per_class
                ),
                len(
                    positions
                )
                // 2,
            )

            if count_each < 2:
                continue

            class_index = len(
                valid_ids
            )

            valid_ids.append(
                int(
                    gt_id
                )
            )

            proto_indices.append(
                positions[
                    :count_each
                ]
            )

            query_indices.append(
                positions[
                    count_each:
                    2
                    * count_each
                ]
            )

            proto_targets.append(
                np.full(
                    count_each,
                    class_index,
                    dtype=np.int64,
                )
            )

            query_targets.append(
                np.full(
                    count_each,
                    class_index,
                    dtype=np.int64,
                )
            )

        if len(
            valid_ids
        ) < 2:
            continue

        groups.append(
            PrototypeGroup(
                source_id=int(
                    source_id
                ),
                gt_ids=tuple(
                    valid_ids
                ),
                proto_indices=torch.from_numpy(
                    np.concatenate(
                        proto_indices
                    )
                ).long(),
                proto_targets=torch.from_numpy(
                    np.concatenate(
                        proto_targets
                    )
                ).long(),
                query_indices=torch.from_numpy(
                    np.concatenate(
                        query_indices
                    )
                ).long(),
                query_targets=torch.from_numpy(
                    np.concatenate(
                        query_targets
                    )
                ).long(),
            )
        )

    return groups


PROTO_GROUPS_D1 = (
    build_prototype_groups(
        CURRENT_D1,
        GT_D1,
        half_per_class=PROTO_HALF_PER_CLASS,
        seed=SEED + 1000,
    )
)

PROTO_GROUPS_D0 = (
    build_prototype_groups(
        CURRENT_D0,
        GT_D0,
        half_per_class=PROTO_HALF_PER_CLASS,
        seed=SEED + 2000,
    )
)

print(
    "Merged D1 groups:",
    [
        (
            g.source_id,
            len(
                g.gt_ids
            ),
            g.gt_ids,
        )
        for g in PROTO_GROUPS_D1
    ],
)

print(
    "Merged D0 groups:",
    [
        (
            g.source_id,
            len(
                g.gt_ids
            ),
            g.gt_ids,
        )
        for g in PROTO_GROUPS_D0
    ],
)

# 6. Verify the center-target intervention before any training

This is a target-design experiment, so first determine whether the alternative target itself resolves the crowded source-9 centers better.

In [ ]:
def physical_coordinate_um(
    index_zyx,
    shape,
    spacing,
):
    index = np.asarray(
        index_zyx,
        dtype=np.float64,
    )
    shape = np.asarray(
        shape,
        dtype=np.float64,
    )
    spacing = np.asarray(
        spacing,
        dtype=np.float64,
    )

    return (
        index
        - 0.5
        * (
            shape - 1.0
        )
    ) * spacing


def source9_peak_metrics_from_probability(
    probability,
    current_labels,
    spacing,
    *,
    max_candidates=None,
):
    probability = np.asarray(
        probability,
        dtype=np.float32,
    )

    source_mask = (
        current_labels
        == SOURCE_ID
    )

    if not source_mask.any():
        return {
            "center_peak_candidates": 0,
            "centers_within_0p5_dref": 0,
            "center_match_mean_um": float("nan"),
            "center_match_median_um": float("nan"),
        }

    coordinates = np.argwhere(
        source_mask
    )

    lo = np.maximum(
        coordinates.min(
            axis=0
        )
        - 2,
        0,
    )
    hi = np.minimum(
        coordinates.max(
            axis=0
        )
        + 3,
        np.asarray(
            probability.shape
        ),
    )

    slices = tuple(
        slice(
            int(a),
            int(b),
        )
        for a, b
        in zip(
            lo,
            hi,
        )
    )

    crop = probability[
        slices
    ]
    source_crop = source_mask[
        slices
    ]

    maxima = ndi.maximum_filter(
        crop,
        size=3,
        mode="constant",
        cval=-np.inf,
    )

    candidate_local = np.argwhere(
        source_crop
        & (
            crop
            >= maxima
        )
    )

    if len(
        candidate_local
    ) == 0:
        return {
            "center_peak_candidates": 0,
            "centers_within_0p5_dref": 0,
            "center_match_mean_um": float("nan"),
            "center_match_median_um": float("nan"),
        }

    candidate_scores = crop[
        tuple(
            candidate_local.T
        )
    ]

    order = np.argsort(
        -candidate_scores
    )

    candidate_local = candidate_local[
        order
    ]
    candidate_scores = candidate_scores[
        order
    ]

    if max_candidates is None:
        max_candidates = max(
            2
            * len(
                source9_gt_ids
            ),
            len(
                source9_gt_ids
            )
            + 4,
        )

    selected_global = []
    selected_coords_um = []

    min_distance_um = (
        0.45
        * dref_um
    )

    for local_index in candidate_local:
        global_index = (
            local_index
            + lo
        )

        coord_um = physical_coordinate_um(
            global_index,
            probability.shape,
            spacing,
        )

        if selected_coords_um:
            distance = np.linalg.norm(
                np.asarray(
                    selected_coords_um
                )
                - coord_um[
                    None, :
                ],
                axis=1,
            )

            if float(
                distance.min()
            ) < min_distance_um:
                continue

        selected_global.append(
            global_index
        )
        selected_coords_um.append(
            coord_um
        )

        if len(
            selected_global
        ) >= int(
            max_candidates
        ):
            break

    selected_coords_um = np.asarray(
        selected_coords_um,
        dtype=np.float32,
    ).reshape(
        -1,
        3,
    )

    top = selected_coords_um[
        : len(
            source9_gt_ids
        )
    ]

    if len(
        top
    ) == 0:
        matched = np.zeros(
            0,
            dtype=np.float32,
        )
    else:
        gt = (
            source9_gt_centers_um
            .numpy()
            .astype(
                np.float32
            )
        )

        distance = np.linalg.norm(
            top[
                :, None, :
            ]
            - gt[
                None, :, :
            ],
            axis=-1,
        )

        rows, cols = (
            linear_sum_assignment(
                distance
            )
        )

        matched = distance[
            rows,
            cols,
        ]

    return {
        "center_peak_candidates": int(
            len(
                selected_coords_um
            )
        ),
        "centers_within_0p5_dref": int(
            np.sum(
                matched
                <= 0.5
                * dref_um
            )
        ),
        "center_match_mean_um": (
            float(
                matched.mean()
            )
            if len(
                matched
            )
            else float(
                "nan"
            )
        ),
        "center_match_median_um": (
            float(
                np.median(
                    matched
                )
            )
            if len(
                matched
            )
            else float(
                "nan"
            )
        ),
    }


original_center_target_np = (
    torch.as_tensor(
        target[
            "center_heatmap"
        ]
    )
    .cpu()
    .numpy()
    .astype(
        np.float32
    )
)

center_target_df = pd.DataFrame([
    {
        "target": "current",
        **source9_peak_metrics_from_probability(
            original_center_target_np,
            current_labels_cpu,
            spacing_native,
        ),
    },
    {
        "target": "max_composed",
        **source9_peak_metrics_from_probability(
            max_center_target_np,
            current_labels_cpu,
            spacing_native,
        ),
    },
])

center_target_df.to_csv(
    RUN_DIR
    / "center_target_intervention.csv",
    index=False,
)

display(center_target_df)

# 7. Tail branch and loss helpers

In [ ]:
class TailBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.stage_e1 = copy.deepcopy(
            TAIL_TEMPLATE[
                "stage_e1"
            ]
        )
        self.stage_e0 = copy.deepcopy(
            TAIL_TEMPLATE[
                "stage_e0"
            ]
        )
        self.dense_heads = copy.deepcopy(
            TAIL_TEMPLATE[
                "dense_heads"
            ]
        )

        # Only used by the deep-internal arms.
        d1_channels = int(
            cfg.spatial.channels[1]
        )

        self.d1_internal_head = nn.Linear(
            d1_channels,
            1,
        )

        nn.init.zeros_(
            self.d1_internal_head.weight
        )
        nn.init.zeros_(
            self.d1_internal_head.bias
        )

    def forward(self):
        d1 = self.stage_e1(
            E2_CACHE,
            SKIP1_CACHE,
            ACQ_CACHE,
        )

        d0 = self.stage_e0(
            d1,
            SKIP0_CACHE,
            ACQ_CACHE,
        )

        dense = self.dense_heads(
            d0
        )

        return {
            "d1": d1,
            "d0": d0,
            "dense": dense,
        }


criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(
    device
)


def current_dense_loss(
    dense_outputs,
    *,
    use_max_center_target=False,
):
    targets = (
        targets_max_center
        if use_max_center_target
        else targets_original
    )

    (
        foreground,
        center_heatmap,
        boundary,
    ) = criterion._dense_losses(
        dense_outputs,
        targets,
    )

    total = (
        float(
            cfg.losses.foreground
        )
        * foreground
        + float(
            cfg.losses.center_heatmap
        )
        * center_heatmap
        + float(
            cfg.losses.boundary
        )
        * boundary
    )

    return {
        "base_total": total,
        "foreground": foreground,
        "center_heatmap": center_heatmap,
        "boundary": boundary,
    }


def sampled_volume_logits(
    logits,
    indices_cpu,
):
    flat = logits.reshape(
        -1
    )

    index = indices_cpu.to(
        device=flat.device,
        non_blocking=True,
    )

    return flat[
        index
    ]


def balanced_binary_loss(
    logits,
    targets,
):
    targets = targets.to(
        device=logits.device,
        dtype=torch.float32,
        non_blocking=True,
    )

    logits = logits.float()

    bce = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
        )
    )

    probability = (
        logits.sigmoid()
    )

    intersection = (
        probability
        * targets
    ).sum()

    dice = (
        1.0
        - (
            2.0
            * intersection
            + 1e-6
        )
        / (
            probability.sum()
            + targets.sum()
            + 1e-6
        )
    )

    return bce + dice


def internal_d0_loss(
    dense_outputs,
):
    logits = sampled_volume_logits(
        dense_outputs[
            "boundary_logits"
        ][0, 0],
        INTERNAL_SAMPLE_D0.indices,
    )

    return balanced_binary_loss(
        logits,
        INTERNAL_SAMPLE_D0.targets,
    )


def gather_feature_rows(
    feature,
    indices_cpu,
):
    # feature [1,C,Z,Y,X] -> selected [N,C]
    flat = (
        feature[
            0
        ]
        .flatten(
            1
        )
        .transpose(
            0,
            1,
        )
    )

    index = indices_cpu.to(
        device=flat.device,
        non_blocking=True,
    )

    return flat[
        index
    ]


def internal_d1_loss(
    branch,
    d1,
):
    feature = gather_feature_rows(
        d1,
        INTERNAL_SAMPLE_D1.indices,
    )

    logits = (
        branch.d1_internal_head(
            feature.float()
        )
        .squeeze(
            -1
        )
    )

    return balanced_binary_loss(
        logits,
        INTERNAL_SAMPLE_D1.targets,
    )


def class_prototypes(
    feature,
    labels,
    n_classes,
):
    feature = F.normalize(
        feature.float(),
        dim=-1,
    )

    prototypes = []

    for class_index in range(
        int(
            n_classes
        )
    ):
        mask = (
            labels
            == class_index
        )

        prototypes.append(
            feature[
                mask
            ].mean(
                dim=0
            )
        )

    return F.normalize(
        torch.stack(
            prototypes,
            dim=0,
        ),
        dim=-1,
    )


def symmetric_prototype_loss(
    feature,
    groups,
    *,
    temperature,
):
    losses = []

    for group in groups:
        proto_feature = gather_feature_rows(
            feature,
            group.proto_indices,
        )

        query_feature = gather_feature_rows(
            feature,
            group.query_indices,
        )

        proto_labels = (
            group.proto_targets.to(
                device=feature.device
            )
        )

        query_labels = (
            group.query_targets.to(
                device=feature.device
            )
        )

        n_classes = len(
            group.gt_ids
        )

        prototype_a = (
            class_prototypes(
                proto_feature,
                proto_labels,
                n_classes,
            )
        )

        prototype_b = (
            class_prototypes(
                query_feature,
                query_labels,
                n_classes,
            )
        )

        query_norm = F.normalize(
            query_feature.float(),
            dim=-1,
        )

        proto_norm = F.normalize(
            proto_feature.float(),
            dim=-1,
        )

        logits_ab = (
            query_norm
            @ prototype_a.T
            / float(
                temperature
            )
        )

        logits_ba = (
            proto_norm
            @ prototype_b.T
            / float(
                temperature
            )
        )

        losses.append(
            0.5
            * (
                F.cross_entropy(
                    logits_ab,
                    query_labels,
                )
                + F.cross_entropy(
                    logits_ba,
                    proto_labels,
                )
            )
        )

    if not losses:
        return feature.sum() * 0.0

    return torch.stack(
        losses
    ).mean()

# 8. Evaluation helpers

Primary metrics:

- source-9 internal-boundary AUC;
- source-9 internal-boundary recall;
- source-9 center recovery;
- source-9 D1/D0 held-out linear instance-mask upper bound;
- original foreground loss as a no-harm signal.

In [ ]:
def binary_auc(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    positive = (
        labels == 1
    )
    negative = (
        labels == 0
    )

    if (
        positive.sum() == 0
        or negative.sum() == 0
    ):
        return float(
            "nan"
        )

    ranks = rankdata(
        scores
    )

    n_pos = int(
        positive.sum()
    )
    n_neg = int(
        negative.sum()
    )

    return float(
        (
            ranks[
                positive
            ].sum()
            - n_pos
            * (
                n_pos + 1
            )
            / 2
        )
        / (
            n_pos
            * n_neg
        )
    )


# Source-9 bbox for cheap D0 output evaluation.
_s9_coords = np.argwhere(
    CURRENT_D0
    == SOURCE_ID
)

S9_D0_LO = np.maximum(
    _s9_coords.min(
        axis=0
    )
    - 2,
    0,
)

S9_D0_HI = np.minimum(
    _s9_coords.max(
        axis=0
    )
    + 3,
    np.asarray(
        D0_SHAPE
    ),
)

S9_D0_SLICES = tuple(
    slice(
        int(a),
        int(b),
    )
    for a, b
    in zip(
        S9_D0_LO,
        S9_D0_HI,
    )
)


def source9_boundary_metrics(
    boundary_logits,
):
    crop_logits = (
        boundary_logits[
            0,
            0,
            S9_D0_SLICES[0],
            S9_D0_SLICES[1],
            S9_D0_SLICES[2],
        ]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    probability = (
        1.0
        / (
            1.0
            + np.exp(
                -crop_logits
            )
        )
    )

    source_crop = (
        CURRENT_D0[
            S9_D0_SLICES
        ]
        == SOURCE_ID
    )

    positive_crop = (
        INTERNAL_RAW_D0[
            S9_D0_SLICES
        ]
        & source_crop
    )

    negative_crop = (
        source_crop
        & ~positive_crop
    )

    positive_scores = (
        probability[
            positive_crop
        ]
    )

    negative_scores = (
        probability[
            negative_crop
        ]
    )

    scores = np.concatenate(
        [
            positive_scores,
            negative_scores,
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(
                    positive_scores
                ),
                dtype=np.int64,
            ),
            np.zeros(
                len(
                    negative_scores
                ),
                dtype=np.int64,
            ),
        ]
    )

    return {
        "source9_internal_auc": (
            binary_auc(
                scores,
                labels,
            )
        ),
        "source9_internal_prob": float(
            positive_scores.mean()
        ),
        "source9_nonboundary_prob": float(
            negative_scores.mean()
        ),
        "source9_internal_recall_0p5": float(
            (
                positive_scores
                >= 0.5
            ).mean()
        ),
    }


def center_metrics_from_logits(
    center_logits,
):
    probability = (
        center_logits[
            0,
            0
        ]
        .detach()
        .float()
        .sigmoid()
        .cpu()
        .numpy()
    )

    return (
        source9_peak_metrics_from_probability(
            probability,
            CURRENT_D0,
            SPACING_D0[
                0
            ].cpu().numpy(),
        )
    )


def source9_group(
    groups,
):
    for group in groups:
        if int(
            group.source_id
        ) == SOURCE_ID:
            return group

    raise RuntimeError(
        "Source-9 prototype group missing."
    )


def heldout_linear_instance_dice(
    feature,
    group,
    *,
    ridge=1e-2,
):
    # Fit one-vs-rest linear logits on proto half, evaluate on query half.
    train_x = gather_feature_rows(
        feature,
        group.proto_indices,
    ).float()

    test_x = gather_feature_rows(
        feature,
        group.query_indices,
    ).float()

    train_y = (
        group.proto_targets.to(
            device=feature.device
        )
    )

    test_y = (
        group.query_targets.to(
            device=feature.device
        )
    )

    n_classes = len(
        group.gt_ids
    )

    mean = train_x.mean(
        dim=0,
        keepdim=True,
    )

    std = train_x.std(
        dim=0,
        keepdim=True,
    ).clamp_min(
        1e-4
    )

    train_x = (
        train_x - mean
    ) / std

    test_x = (
        test_x - mean
    ) / std

    train_design = torch.cat(
        [
            train_x,
            torch.ones(
                (
                    len(
                        train_x
                    ),
                    1,
                ),
                device=train_x.device,
            ),
        ],
        dim=-1,
    )

    test_design = torch.cat(
        [
            test_x,
            torch.ones(
                (
                    len(
                        test_x
                    ),
                    1,
                ),
                device=test_x.device,
            ),
        ],
        dim=-1,
    )

    target_matrix = (
        F.one_hot(
            train_y,
            num_classes=n_classes,
        )
        .float()
        * 2.0
        - 1.0
    )

    eye = torch.eye(
        train_design.shape[
            1
        ],
        device=train_design.device,
        dtype=torch.float32,
    )

    eye[
        -1,
        -1,
    ] = 0.0

    gram = (
        train_design.T
        @ train_design
        + float(
            ridge
        )
        * eye
    )

    rhs = (
        train_design.T
        @ target_matrix
    )

    weight = torch.linalg.solve(
        gram,
        rhs,
    )

    logits = (
        test_design
        @ weight
    )

    probability = (
        logits.sigmoid()
    )

    target_onehot = F.one_hot(
        test_y,
        num_classes=n_classes,
    ).float()

    intersection = (
        probability
        * target_onehot
    ).sum(
        dim=0
    )

    dice = (
        2.0
        * intersection
        + 1e-6
    ) / (
        probability.sum(
            dim=0
        )
        + target_onehot.sum(
            dim=0
        )
        + 1e-6
    )

    accuracy = (
        logits.argmax(
            dim=-1
        )
        == test_y
    ).float().mean()

    return {
        "soft_dice": float(
            dice.mean()
            .detach()
            .cpu()
        ),
        "accuracy": float(
            accuracy.detach()
            .cpu()
        ),
    }


S9_GROUP_D1 = source9_group(
    PROTO_GROUPS_D1
)

S9_GROUP_D0 = source9_group(
    PROTO_GROUPS_D0
)

# 9. Define independent experiment arms

In [ ]:
ARM_SPECS = {
    "baseline": {
        "internal_d0": False,
        "internal_d1": False,
        "prototype": False,
        "max_center": False,
    },
    "internal_boundary": {
        "internal_d0": True,
        "internal_d1": False,
        "prototype": False,
        "max_center": False,
    },
    "deep_internal_boundary": {
        "internal_d0": True,
        "internal_d1": True,
        "prototype": False,
        "max_center": False,
    },
    "instance_prototype": {
        "internal_d0": False,
        "internal_d1": False,
        "prototype": True,
        "max_center": False,
    },
    "max_center_target": {
        "internal_d0": False,
        "internal_d1": False,
        "prototype": False,
        "max_center": True,
    },
    "combined": {
        "internal_d0": True,
        "internal_d1": True,
        "prototype": True,
        "max_center": True,
    },
}

display(
    pd.DataFrame(
        ARM_SPECS
    ).T
)

# 10. Objective and evaluation

In [ ]:
def arm_objective(
    branch,
    outputs,
    spec,
):
    base = current_dense_loss(
        outputs[
            "dense"
        ],
        use_max_center_target=bool(
            spec[
                "max_center"
            ]
        ),
    )

    zero = (
        base[
            "base_total"
        ]
        * 0.0
    )

    loss_internal_d0 = zero
    loss_internal_d1 = zero
    loss_proto_d1 = zero
    loss_proto_d0 = zero

    if spec[
        "internal_d0"
    ]:
        loss_internal_d0 = (
            internal_d0_loss(
                outputs[
                    "dense"
                ]
            )
        )

    if spec[
        "internal_d1"
    ]:
        loss_internal_d1 = (
            internal_d1_loss(
                branch,
                outputs[
                    "d1"
                ],
            )
        )

    if spec[
        "prototype"
    ]:
        loss_proto_d1 = (
            symmetric_prototype_loss(
                outputs[
                    "d1"
                ],
                PROTO_GROUPS_D1,
                temperature=(
                    PROTO_TEMPERATURE
                ),
            )
        )

        loss_proto_d0 = (
            symmetric_prototype_loss(
                outputs[
                    "d0"
                ],
                PROTO_GROUPS_D0,
                temperature=(
                    PROTO_TEMPERATURE
                ),
            )
        )

    total = (
        base[
            "base_total"
        ]
        + float(
            LAMBDA_INTERNAL_D0
        )
        * loss_internal_d0
        + float(
            LAMBDA_INTERNAL_D1
        )
        * loss_internal_d1
        + float(
            LAMBDA_PROTO_D1
        )
        * loss_proto_d1
        + float(
            LAMBDA_PROTO_D0
        )
        * loss_proto_d0
    )

    return {
        "loss": total,
        "base_total": (
            base[
                "base_total"
            ]
        ),
        "foreground": (
            base[
                "foreground"
            ]
        ),
        "center_heatmap": (
            base[
                "center_heatmap"
            ]
        ),
        "boundary": (
            base[
                "boundary"
            ]
        ),
        "internal_d0": (
            loss_internal_d0
        ),
        "internal_d1": (
            loss_internal_d1
        ),
        "prototype_d1": (
            loss_proto_d1
        ),
        "prototype_d0": (
            loss_proto_d0
        ),
    }


@torch.no_grad()
def evaluate_branch(
    branch,
    *,
    arm,
    step,
):
    branch.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = branch()

        # Evaluate all arms against the ORIGINAL dense target
        # so the base-loss comparison remains apples-to-apples.
        base_eval = current_dense_loss(
            outputs[
                "dense"
            ],
            use_max_center_target=False,
        )

    boundary = (
        source9_boundary_metrics(
            outputs[
                "dense"
            ][
                "boundary_logits"
            ]
        )
    )

    center = center_metrics_from_logits(
        outputs[
            "dense"
        ][
            "center_heatmap_logits"
        ]
    )

    d1_linear = (
        heldout_linear_instance_dice(
            outputs[
                "d1"
            ],
            S9_GROUP_D1,
        )
    )

    d0_linear = (
        heldout_linear_instance_dice(
            outputs[
                "d0"
            ],
            S9_GROUP_D0,
        )
    )

    result = {
        "arm": arm,
        "step": int(
            step
        ),
        "original_base_loss": float(
            base_eval[
                "base_total"
            ].detach().cpu()
        ),
        "foreground_loss": float(
            base_eval[
                "foreground"
            ].detach().cpu()
        ),
        "center_loss": float(
            base_eval[
                "center_heatmap"
            ].detach().cpu()
        ),
        "boundary_loss": float(
            base_eval[
                "boundary"
            ].detach().cpu()
        ),
        **boundary,
        **center,
        "d1_oracle_linear_dice": (
            d1_linear[
                "soft_dice"
            ]
        ),
        "d1_linear_accuracy": (
            d1_linear[
                "accuracy"
            ]
        ),
        "d0_oracle_linear_dice": (
            d0_linear[
                "soft_dice"
            ]
        ),
        "d0_linear_accuracy": (
            d0_linear[
                "accuracy"
            ]
        ),
    }

    del outputs
    return result

# 11. One common zero-step baseline

All experiment arms start from identical tail weights, so the learned-model metrics should be identical at step 0.

The only newly initialized parameter is the D1 auxiliary internal-boundary readout, which does not affect the base forward.

In [ ]:
def build_fresh_branch():
    torch.manual_seed(
        SEED + 9001
    )
    torch.cuda.manual_seed_all(
        SEED + 9001
    )

    branch = TailBranch().to(
        device
    )

    return branch


zero_branch = build_fresh_branch()

zero_metrics = evaluate_branch(
    zero_branch,
    arm="common_step0",
    step=0,
)

display(
    pd.DataFrame(
        [zero_metrics]
    )
)

del zero_branch
gc.collect()
torch.cuda.empty_cache()

# 12. Three-step independent micro screen

Every arm is rebuilt from the same step-30 tail.

The screen records:

- objective components;
- gradient norm;
- runtime;
- peak CUDA memory;
- failed spatial metrics at steps 0, 1, and 3.

No state is shared between arms.

In [ ]:
def cpu_state_dict(
    module,
):
    return {
        key: value.detach()
        .cpu()
        .clone()
        for key, value
        in module.state_dict().items()
    }


def run_arm(
    arm,
    spec,
    *,
    total_steps,
    initial_state=None,
    starting_step=0,
):
    branch = build_fresh_branch()

    if initial_state is not None:
        branch.load_state_dict(
            initial_state,
            strict=True,
        )

    optimizer = torch.optim.AdamW(
        branch.parameters(),
        lr=float(
            MICRO_LR
        ),
        weight_decay=float(
            cfg.training.weight_decay
        ),
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=1024.0,
    )

    metric_rows = []
    train_rows = []

    branch_start = time.perf_counter()

    eval_points = set(
        [
            starting_step,
            starting_step + 1,
            total_steps,
        ]
    )

    if (
        starting_step == 0
        and total_steps >= 3
    ):
        eval_points.add(
            3
        )

    for step in range(
        starting_step,
        total_steps + 1,
    ):
        if step in eval_points:
            metric_rows.append(
                evaluate_branch(
                    branch,
                    arm=arm,
                    step=step,
                )
            )

        if step == total_steps:
            break

        branch.train()
        optimizer.zero_grad(
            set_to_none=True
        )

        torch.cuda.reset_peak_memory_stats()
        train_start = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = branch()

            objective = arm_objective(
                branch,
                outputs,
                spec,
            )

        scaler.scale(
            objective[
                "loss"
            ]
        ).backward()

        scaler.unscale_(
            optimizer
        )

        grad_sq = 0.0
        finite_gradients = True

        for parameter in branch.parameters():
            if parameter.grad is None:
                continue

            grad = parameter.grad.detach()

            finite_gradients = (
                finite_gradients
                and bool(
                    torch.isfinite(
                        grad
                    ).all()
                )
            )

            grad_sq += float(
                grad.float()
                .square()
                .sum()
                .detach()
                .cpu()
            )

        grad_norm = math.sqrt(
            max(
                grad_sq,
                0.0,
            )
        )

        torch.nn.utils.clip_grad_norm_(
            branch.parameters(),
            float(
                cfg.training.max_grad_norm
            ),
        )

        scaler.step(
            optimizer
        )
        scaler.update()

        train_rows.append({
            "arm": arm,
            "from_step": int(
                step
            ),
            "to_step": int(
                step + 1
            ),
            "loss": float(
                objective[
                    "loss"
                ].detach().cpu()
            ),
            "base_total": float(
                objective[
                    "base_total"
                ].detach().cpu()
            ),
            "foreground": float(
                objective[
                    "foreground"
                ].detach().cpu()
            ),
            "center_heatmap": float(
                objective[
                    "center_heatmap"
                ].detach().cpu()
            ),
            "boundary": float(
                objective[
                    "boundary"
                ].detach().cpu()
            ),
            "internal_d0": float(
                objective[
                    "internal_d0"
                ].detach().cpu()
            ),
            "internal_d1": float(
                objective[
                    "internal_d1"
                ].detach().cpu()
            ),
            "prototype_d1": float(
                objective[
                    "prototype_d1"
                ].detach().cpu()
            ),
            "prototype_d0": float(
                objective[
                    "prototype_d0"
                ].detach().cpu()
            ),
            "grad_norm_before_clip": float(
                grad_norm
            ),
            "finite_gradients": bool(
                finite_gradients
            ),
            "step_seconds": float(
                time.perf_counter()
                - train_start
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        })

        del outputs
        del objective

    final_state = cpu_state_dict(
        branch
    )

    elapsed = (
        time.perf_counter()
        - branch_start
    )

    del branch
    del optimizer
    del scaler

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "metrics": pd.DataFrame(
            metric_rows
        ),
        "training": pd.DataFrame(
            train_rows
        ),
        "state": final_state,
        "elapsed_s": float(
            elapsed
        ),
    }


SCREEN_RUNS = {}
metric_frames = []
training_frames = []

for arm, spec in ARM_SPECS.items():
    print(
        "\n"
        + "=" * 72
    )
    print(
        "SCREEN:",
        arm
    )
    print(
        "=" * 72
    )

    result = run_arm(
        arm,
        spec,
        total_steps=SCREEN_STEPS,
        initial_state=None,
        starting_step=0,
    )

    SCREEN_RUNS[
        arm
    ] = result

    metric_frames.append(
        result[
            "metrics"
        ]
    )

    training_frames.append(
        result[
            "training"
        ]
    )

    display(
        result[
            "metrics"
        ][
            [
                "arm",
                "step",
                "source9_internal_auc",
                "source9_internal_recall_0p5",
                "centers_within_0p5_dref",
                "d1_oracle_linear_dice",
                "d0_oracle_linear_dice",
                "foreground_loss",
            ]
        ]
    )

    print(
        "elapsed:",
        round(
            result[
                "elapsed_s"
            ],
            2,
        ),
        "s",
    )


screen_metrics_df = pd.concat(
    metric_frames,
    ignore_index=True,
)

screen_training_df = pd.concat(
    training_frames,
    ignore_index=True,
)

screen_metrics_df.to_csv(
    RUN_DIR
    / "screen_metrics.csv",
    index=False,
)

screen_training_df.to_csv(
    RUN_DIR
    / "screen_training.csv",
    index=False,
)

print(
    "\nAll 3-step screens complete."
)

# 13. Compare the six mechanisms at step 3

In [ ]:
step3 = (
    screen_metrics_df[
        screen_metrics_df[
            "step"
        ]
        == SCREEN_STEPS
    ]
    .copy()
)

baseline0 = zero_metrics

# Diagnostic score is only for selecting two arms to extend.
# Raw metrics remain the actual decision evidence.
step3[
    "diagnostic_score"
] = (
    2.0
    * (
        step3[
            "source9_internal_auc"
        ]
        - float(
            baseline0[
                "source9_internal_auc"
            ]
        )
    )
    + 1.0
    * (
        step3[
            "d1_oracle_linear_dice"
        ]
        - float(
            baseline0[
                "d1_oracle_linear_dice"
            ]
        )
    )
    + 0.5
    * (
        step3[
            "d0_oracle_linear_dice"
        ]
        - float(
            baseline0[
                "d0_oracle_linear_dice"
            ]
        )
    )
    + 0.03
    * (
        step3[
            "centers_within_0p5_dref"
        ]
        - int(
            baseline0[
                "centers_within_0p5_dref"
            ]
        )
    )
    - 0.25
    * np.maximum(
        0.0,
        (
            step3[
                "foreground_loss"
            ]
            / max(
                float(
                    baseline0[
                        "foreground_loss"
                    ]
                ),
                1e-8,
            )
            - 1.0
        ),
    )
)

step3 = step3.sort_values(
    "diagnostic_score",
    ascending=False,
)

display(
    step3[
        [
            "arm",
            "source9_internal_auc",
            "source9_internal_recall_0p5",
            "centers_within_0p5_dref",
            "center_match_median_um",
            "d1_oracle_linear_dice",
            "d1_linear_accuracy",
            "d0_oracle_linear_dice",
            "d0_linear_accuracy",
            "foreground_loss",
            "diagnostic_score",
        ]
    ]
)

step3.to_csv(
    RUN_DIR
    / "step3_arm_comparison.csv",
    index=False,
)

# 14. Extend only the two best non-baseline mechanisms to step 5

This is deliberately bounded. We do not extend every arm.

In [ ]:
candidate_step3 = step3[
    step3[
        "arm"
    ]
    != "baseline"
]

TOP_ARMS = (
    candidate_step3[
        "arm"
    ]
    .head(
        2
    )
    .tolist()
)

print(
    "Extending:",
    TOP_ARMS,
)

extension_metric_frames = []
extension_training_frames = []
EXTENSION_RUNS = {}

for arm in TOP_ARMS:
    spec = ARM_SPECS[
        arm
    ]

    print(
        "\n"
        + "=" * 72
    )
    print(
        "EXTEND:",
        arm,
        f"{SCREEN_STEPS} → {EXTEND_TO_STEP}"
    )
    print(
        "=" * 72
    )

    result = run_arm(
        arm,
        spec,
        total_steps=EXTEND_TO_STEP,
        initial_state=(
            SCREEN_RUNS[
                arm
            ][
                "state"
            ]
        ),
        starting_step=SCREEN_STEPS,
    )

    EXTENSION_RUNS[
        arm
    ] = result

    extension_metric_frames.append(
        result[
            "metrics"
        ]
    )

    extension_training_frames.append(
        result[
            "training"
        ]
    )

    display(
        result[
            "metrics"
        ][
            [
                "arm",
                "step",
                "source9_internal_auc",
                "source9_internal_recall_0p5",
                "centers_within_0p5_dref",
                "d1_oracle_linear_dice",
                "d0_oracle_linear_dice",
                "foreground_loss",
            ]
        ]
    )


if extension_metric_frames:
    extension_metrics_df = pd.concat(
        extension_metric_frames,
        ignore_index=True,
    )

    extension_training_df = pd.concat(
        extension_training_frames,
        ignore_index=True,
    )
else:
    extension_metrics_df = pd.DataFrame()
    extension_training_df = pd.DataFrame()

extension_metrics_df.to_csv(
    RUN_DIR
    / "extension_metrics.csv",
    index=False,
)

extension_training_df.to_csv(
    RUN_DIR
    / "extension_training.csv",
    index=False,
)

# 15. Final mechanism decision

Interpretation is based on **which failed quantity moves**, not on total objective alone.

### Internal-boundary mechanism succeeds if

- source-9 internal-boundary AUC rises materially;
- recall rises;
- foreground loss does not collapse.

### Instance-prototype mechanism succeeds if

- D1/D0 held-out linear instance Dice rises materially.

### Center-target mechanism succeeds if

- the target itself resolves more centers;
- predicted center recovery starts moving within a few steps.

### Combined succeeds if

it improves at least two of the above without damaging the foreground objective.

In [ ]:
all_final_rows = []

# Baseline and non-extended arms use screen step 3.
for arm in ARM_SPECS:
    if arm in TOP_ARMS:
        rows = extension_metrics_df[
            (
                extension_metrics_df[
                    "arm"
                ]
                == arm
            )
            & (
                extension_metrics_df[
                    "step"
                ]
                == EXTEND_TO_STEP
            )
        ]

        if len(
            rows
        ):
            all_final_rows.append(
                rows.iloc[
                    0
                ].to_dict()
            )
            continue

    rows = screen_metrics_df[
        (
            screen_metrics_df[
                "arm"
            ]
            == arm
        )
        & (
            screen_metrics_df[
                "step"
            ]
            == SCREEN_STEPS
        )
    ]

    if len(
        rows
    ):
        all_final_rows.append(
            rows.iloc[
                0
            ].to_dict()
        )


final_df = pd.DataFrame(
    all_final_rows
)

base_auc = float(
    baseline0[
        "source9_internal_auc"
    ]
)
base_d1 = float(
    baseline0[
        "d1_oracle_linear_dice"
    ]
)
base_d0 = float(
    baseline0[
        "d0_oracle_linear_dice"
    ]
)
base_centers = int(
    baseline0[
        "centers_within_0p5_dref"
    ]
)
base_fg = float(
    baseline0[
        "foreground_loss"
    ]
)

final_df[
    "delta_internal_auc"
] = (
    final_df[
        "source9_internal_auc"
    ]
    - base_auc
)

final_df[
    "delta_d1_oracle_dice"
] = (
    final_df[
        "d1_oracle_linear_dice"
    ]
    - base_d1
)

final_df[
    "delta_d0_oracle_dice"
] = (
    final_df[
        "d0_oracle_linear_dice"
    ]
    - base_d0
)

final_df[
    "delta_centers"
] = (
    final_df[
        "centers_within_0p5_dref"
    ]
    - base_centers
)

final_df[
    "foreground_loss_ratio"
] = (
    final_df[
        "foreground_loss"
    ]
    / max(
        base_fg,
        1e-8,
    )
)

final_df = final_df.sort_values(
    [
        "delta_internal_auc",
        "delta_d1_oracle_dice",
    ],
    ascending=False,
)

display(
    final_df[
        [
            "arm",
            "step",
            "source9_internal_auc",
            "delta_internal_auc",
            "source9_internal_recall_0p5",
            "centers_within_0p5_dref",
            "delta_centers",
            "d1_oracle_linear_dice",
            "delta_d1_oracle_dice",
            "d0_oracle_linear_dice",
            "delta_d0_oracle_dice",
            "foreground_loss_ratio",
        ]
    ]
)

final_df.to_csv(
    RUN_DIR
    / "final_arm_comparison.csv",
    index=False,
)

In [ ]:
findings = []

# Target-side center experiment.
current_target_centers = int(
    center_target_df[
        center_target_df[
            "target"
        ]
        == "current"
    ][
        "centers_within_0p5_dref"
    ].iloc[
        0
    ]
)

max_target_centers = int(
    center_target_df[
        center_target_df[
            "target"
        ]
        == "max_composed"
    ][
        "centers_within_0p5_dref"
    ].iloc[
        0
    ]
)

if (
    max_target_centers
    > current_target_centers
):
    findings.append(
        f"Center target design is causally relevant: target recovery improves "
        f"{current_target_centers}/{len(source9_gt_ids)} → "
        f"{max_target_centers}/{len(source9_gt_ids)} before training."
    )
else:
    findings.append(
        "Max-composed center targets do not improve source-9 target peak recovery; "
        "do not prioritize this change."
    )


def final_row(arm):
    rows = final_df[
        final_df[
            "arm"
        ]
        == arm
    ]

    return (
        rows.iloc[
            0
        ]
        if len(
            rows
        )
        else None
    )


internal_row = final_row(
    "internal_boundary"
)
deep_row = final_row(
    "deep_internal_boundary"
)
proto_row = final_row(
    "instance_prototype"
)
combined_row = final_row(
    "combined"
)


if (
    internal_row is not None
    and float(
        internal_row[
            "delta_internal_auc"
        ]
    )
    >= 0.05
):
    findings.append(
        "Explicit independently normalized internal-boundary supervision works: "
        f"AUC gain={float(internal_row['delta_internal_auc']):+.3f}."
    )
else:
    findings.append(
        "Shared-head internal-boundary supervision does not move source-9 AUC by +0.05 "
        "within the micro screen."
    )


if (
    deep_row is not None
    and internal_row is not None
    and float(
        deep_row[
            "delta_internal_auc"
        ]
    )
    > float(
        internal_row[
            "delta_internal_auc"
        ]
    )
    + 0.02
):
    findings.append(
        "D1 deep internal-boundary supervision adds material value beyond D0-only "
        "rebalancing; preserve instance-boundary information through stage_e1."
    )


if (
    proto_row is not None
    and (
        float(
            proto_row[
                "delta_d1_oracle_dice"
            ]
        )
        >= 0.05
        or float(
            proto_row[
                "delta_d0_oracle_dice"
            ]
        )
        >= 0.05
    )
):
    findings.append(
        "Instance-prototype supervision rapidly makes D1/D0 more linearly instance-separable."
    )
else:
    findings.append(
        "Prototype supervision does not raise the held-out instance-mask upper bound "
        "by +0.05 within the micro screen."
    )


if combined_row is not None:
    combined_multi_signal = (
        int(
            float(
                combined_row[
                    "delta_internal_auc"
                ]
            )
            >= 0.05
        )
        + int(
            float(
                combined_row[
                    "delta_d1_oracle_dice"
                ]
            )
            >= 0.05
        )
        + int(
            float(
                combined_row[
                    "delta_d0_oracle_dice"
                ]
            )
            >= 0.05
        )
        + int(
            int(
                combined_row[
                    "delta_centers"
                ]
            )
            >= 1
        )
    )

    combined_no_harm = (
        float(
            combined_row[
                "foreground_loss_ratio"
            ]
        )
        <= 1.10
    )

    if (
        combined_multi_signal >= 2
        and combined_no_harm
    ):
        findings.append(
            "Combined spatial supervision is GREEN: at least two failed spatial signals "
            "improve without >10% foreground-loss regression."
        )
        verdict = "GREEN"
    elif combined_multi_signal >= 1:
        findings.append(
            "Combined spatial supervision is YELLOW: at least one failed signal moves, "
            "but the complete spatial fix is not yet established."
        )
        verdict = "YELLOW"
    else:
        findings.append(
            "Combined spatial supervision is RED: the proposed losses do not move the "
            "failed spatial representation within five tail steps."
        )
        verdict = "RED"
else:
    verdict = "UNKNOWN"


print(
    "=" * 82
)
print(
    "NOTEBOOK 21 — SPATIAL FIX MICRO-EXPERIMENT VERDICT"
)
print(
    "=" * 82
)
print(
    "Verdict:",
    verdict
)

for index, item in enumerate(
    findings,
    start=1,
):
    print(
        f"{index}. {item}"
    )


report = {
    "verdict": verdict,
    "screen_steps": SCREEN_STEPS,
    "extended_to_step": EXTEND_TO_STEP,
    "top_arms_extended": TOP_ARMS,
    "center_target_current_recovery": (
        current_target_centers
    ),
    "center_target_max_recovery": (
        max_target_centers
    ),
    "findings": findings,
}

with (
    RUN_DIR
    / "mechanism_verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

# 16. Compact plots

In [ ]:
plot_df = final_df.set_index(
    "arm"
)

ax = plot_df[
    "source9_internal_auc"
].plot(
    kind="bar",
    figsize=(9, 4),
)
ax.axhline(
    base_auc,
    linestyle="--",
)
ax.set_ylabel(
    "source-9 internal-boundary AUC"
)
ax.set_title(
    "Micro experiment: internal-boundary separation"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

ax = plot_df[
    [
        "d1_oracle_linear_dice",
        "d0_oracle_linear_dice",
    ]
].plot(
    kind="bar",
    figsize=(10, 4),
)
ax.set_ylabel(
    "held-out linear instance soft Dice"
)
ax.set_title(
    "Micro experiment: instance separability"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

ax = plot_df[
    "centers_within_0p5_dref"
].plot(
    kind="bar",
    figsize=(9, 4),
)
ax.set_ylabel(
    "source-9 centers within 0.5 dref"
)
ax.set_title(
    "Micro experiment: center recovery"
)
plt.xticks(
    rotation=35,
    ha="right",
)
plt.tight_layout()
plt.show()

# 17. Save timing / artifact manifest

Do not launch a full staged overfit from this notebook.

The next step is to inspect the mechanism verdict and implement only the mechanisms that demonstrated causal movement.

In [ ]:
timing_rows = []

for arm, result in SCREEN_RUNS.items():
    timing_rows.append({
        "phase": "screen",
        "arm": arm,
        "seconds": float(
            result[
                "elapsed_s"
            ]
        ),
    })

for arm, result in EXTENSION_RUNS.items():
    timing_rows.append({
        "phase": "extension",
        "arm": arm,
        "seconds": float(
            result[
                "elapsed_s"
            ]
        ),
    })

timing_df = pd.DataFrame(
    timing_rows
)

timing_df.to_csv(
    RUN_DIR
    / "timing.csv",
    index=False,
)

manifest = {
    "step30_checkpoint": str(
        STEP30_CHECKPOINT
    ),
    "cache_seconds": float(
        cache_seconds
    ),
    "screen_steps": int(
        SCREEN_STEPS
    ),
    "extension_step": int(
        EXTEND_TO_STEP
    ),
    "top_arms_extended": (
        TOP_ARMS
    ),
    "files": [
        path.name
        for path in sorted(
            RUN_DIR.glob("*")
        )
    ],
}

with (
    RUN_DIR
    / "manifest.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

display(timing_df)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)

gc.collect()
torch.cuda.empty_cache()

print(
    "Notebook 21 complete. No full overfit was run."
)

# Decision guide

## If `internal_boundary` wins

Implement:

```text
explicit internal cell-cell boundary target
+
independently normalized internal-boundary loss
```

Do not rely on one globally averaged mixed outer+internal boundary objective.

## If `deep_internal_boundary` clearly beats `internal_boundary`

Also add D1-scale auxiliary internal-boundary supervision.

This directly addresses the measured E2 → D1 information loss.

## If `instance_prototype` wins

The main missing signal is not merely boundary classification.

The spatial features need direct **instance-aware representation supervision**. Implement a production-friendly instance embedding/prototype/discriminative loss during spatial training.

## If `max_center_target` wins

Change center target generation to max-composed per-instance Gaussians.

## If `combined` wins while the individual arms each move different metrics

Implement the changes together, but still validate each new source component with targeted regression tests.

## If every arm is RED

Do not train longer.

The next fix should move from supervision toward the E2 → D1 decoder transformation itself, because the measured supervision interventions cannot make the existing tail exploit E2 within a few steps.